# TrafficVision — Entrenamiento YOLOv8n
**Detección de placas vehiculares ecuatorianas**

Pasos:
1. Verificar GPU
2. Instalar dependencias
3. Montar Google Drive
4. Preparar datasets
5. Entrenar modelos
6. Descargar resultados

In [ ]:
# ── CELDA ANTI-DESCONEXIÓN ────────────────────────────────────
# Ejecuta esto en la CONSOLA DEL NAVEGADOR (F12 → Consola):
# function keepAlive() { document.querySelector("#top-toolbar").click(); setTimeout(keepAlive, 60000); } keepAlive();

# También instala esto en Colab para auto-reconectar:
import time, threading

def heartbeat():
    while True:
        time.sleep(45)
        try:
            from google.colab import output
            output.eval_js("document.querySelector("#top-toolbar").click()")
        except:
            pass

thread = threading.Thread(target=heartbeat, daemon=True)
thread.start()
print("✅ Anti-desconexión activo")

In [ ]:
# ── CELDA 1: Verificar GPU ─────────────────────────────────────────
!nvidia-smi
import torch
print(f'\nCUDA disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')

In [ ]:
# ── CELDA 2: Instalar dependencias ────────────────────────────────
!pip install ultralytics -q
from ultralytics import YOLO
print('✅ Ultralytics instalado')

In [ ]:
# ── CELDA 3: Montar Google Drive ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

In [ ]:
# ── CELDA 4: Verificar estructura de datasets ─────────────────────
import os

# Ruta base en tu Google Drive
# IMPORTANTE: ajusta esta ruta según donde subiste los datasets
DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'

datasets = {
    'global':   f'{DRIVE_BASE}/license-plates',
    'ecuador1': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1',
    'ecuador2': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2',
    'ecuador4': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4',
}

for name, path in datasets.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌'
    if exists:
        train_count = len(os.listdir(f'{path}/train/images')) if os.path.exists(f'{path}/train/images') else 0
        print(f'{status} {name}: {train_count} imágenes de entrenamiento')
    else:
        print(f'{status} {name}: NO ENCONTRADO en {path}')

In [ ]:
# ── CELDA 5: Crear data.yaml para combined_all ────────────────────
import yaml

DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'

data = {
    'train': [
        f'{DRIVE_BASE}/license-plates/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    'val':  f'{DRIVE_BASE}/license-plates/valid/images',
    'test': f'{DRIVE_BASE}/license-plates/test/images',
    'nc':   1,
    'names': ['license plate'],
}

yaml_path = '/content/data_combined_all.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('✅ data_combined_all.yaml creado')
print(f'   Train: {len(data["train"])} carpetas')
for p in data['train']:
    count = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f'   - {p.split("/")[-3]}/{p.split("/")[-2]}: {count} imgs')

In [ ]:
# ── CELDA 6: Entrenar combined_all (global + ecuador) ─────────────
# ⏱️ Estimado con GPU T4: 30-60 minutos

model = YOLO('yolov8n.pt')

results = model.train(
    data     = '/content/data_combined_all.yaml',
    epochs   = 100,       # más épocas que en CPU
    imgsz    = 640,
    batch    = 32,        # batch más grande con GPU
    name     = 'yolov8n_combined_all',
    project  = '/content/drive/MyDrive/TrafficVision/runs',
    patience = 15,
    save     = True,
    plots    = True,
    device   = 0,         # GPU
    amp      = True,      # Mixed precision — más rápido en GPU
    save_period = 5,       # guardar cada 5 épocas
)

print('\n✅ Entrenamiento completado')

In [ ]:
last_pt = '/content/drive/MyDrive/TrafficVision/runs/yolov8n_combined_all/weights/last.pt'

model = YOLO(last_pt)        # ← cargar last.pt en vez de yolov8n.pt

results = model.train(
    data        = '/content/data_combined_all.yaml',
    epochs      = 100,
    imgsz       = 640,
    batch       = 32,
    name        = 'yolov8n_combined_all',
    project     = '/content/drive/MyDrive/TrafficVision/runs',
    patience    = 15,
    save        = True,
    plots       = True,
    device      = 0,
    amp         = True,
    save_period = 5,
    resume      = True,      # ← reanudar desde época 55
)

print('\n✅ Entrenamiento completado')

In [ ]:
# ── CELDA 7 (OPCIONAL): Entrenar solo Ecuador ─────────────────────
import yaml

DRIVE_BASE = '/content/drive/MyDrive/TrafficVision/datasets'

data_ec = {
    'train': [
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    'val':  f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/valid/images',
    'test': f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/test/images',
    'nc':   1,
    'names': ['license plate'],
}

with open('/content/data_ecuador.yaml', 'w') as f:
    yaml.dump(data_ec, f, default_flow_style=False)

model_ec = YOLO('yolov8n.pt')
model_ec.train(
    data     = '/content/data_ecuador.yaml',
    epochs   = 100,
    imgsz    = 640,
    batch    = 32,
    name     = 'yolov8n_ecuador_combined',
    project  = '/content/drive/MyDrive/TrafficVision/runs',
    patience = 15,
    save     = True,
    plots    = True,
    device   = 0,
    amp      = True,
)

print('\n✅ Entrenamiento Ecuador completado')

In [ ]:
# ── CELDA REANUDAR ENTRENAMIENTO ──────────────────────────────
# Usa esto si Colab se desconectó y quieres continuar
import glob, os

RUNS_DIR = "/content/drive/MyDrive/TrafficVision/runs"

# Buscar último checkpoint guardado
last_models = glob.glob(f"{RUNS_DIR}/**/weights/last.pt", recursive=True)

if last_models:
    last_pt = last_models[0]
    print(f"✅ Checkpoint encontrado: {last_pt}")
    size = os.path.getsize(last_pt) / (1024*1024)
    print(f"   Tamaño: {size:.1f} MB")

    # Reanudar desde el último checkpoint
    model = YOLO(last_pt)
    model.train(
        data        = "/content/data_combined_all.yaml",
        epochs      = 100,
        imgsz       = 640,
        batch       = 32,
        name        = "yolov8n_combined_all",
        project     = f"{RUNS_DIR}",
        patience    = 15,
        save        = True,
        save_period = 5,     # guardar cada 5 épocas
        plots       = True,
        device      = 0,
        amp         = True,
        resume      = True,  # ← clave para reanudar
    )
else:
    print("❌ No se encontró checkpoint. Ejecuta la Celda 6 desde el inicio.")

In [ ]:
# ── CELDA 8: Ver métricas finales ────────────────────────────────
import glob

runs_dir = '/content/drive/MyDrive/TrafficVision/runs'
best_models = glob.glob(f'{runs_dir}/**/weights/best.pt', recursive=True)

print('Modelos entrenados:')
for m in best_models:
    size = os.path.getsize(m) / (1024*1024)
    print(f'  ✅ {m.split("/")[-3]} — {size:.1f} MB')

# Evaluar el mejor modelo
if best_models:
    model_eval = YOLO(best_models[0])
    metrics = model_eval.val(
        data   = '/content/data_combined_all.yaml',
        imgsz  = 640,
        device = 0,
    )
    print(f'\n── Métricas ───────────────────────')
    print(f'  mAP@50:    {metrics.box.map50:.4f}')
    print(f'  mAP@50-95: {metrics.box.map:.4f}')
    print(f'  Precisión: {metrics.box.mp:.4f}')
    print(f'  Recall:    {metrics.box.mr:.4f}')